# PREPARE DATA

In [51]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

Extract from source, clean and export for further usage

Final output for MN income data: MN income data = mn_income_clean.csv

Final output for MN Crime data: MN crime data = mn_crime_clean.csv

## _1. MN Income Data_
Source: https://data.census.gov/

https://data.census.gov/table/ACSST1Y2024.S1901?t=Income+(Households,+Families,+Individuals):Income+and+Poverty&g=010XX00US$0500000_040XX00US27,27$0500000&y=2024


In [55]:
income_2020 = pd.read_csv("ACSST5Y2020.S1901-2025-11-22T145514.csv", delimiter=',')
income_2021 = pd.read_csv("ACSST5Y2021.S1901-2025-11-22T145449.csv", delimiter=',')
income_2022 = pd.read_csv("ACSST5Y2022.S1901-2025-11-22T145407.csv", delimiter=',')
income_2023 = pd.read_csv("ACSST5Y2023.S1901-2025-11-22T145322.csv", delimiter=',')
income_2024 = pd.read_csv("ACSST1Y2024.S1901-2025-11-22T145242.csv", delimiter=',')
income_2020['year'] = 2020
income_2021['year'] = 2021
income_2022['year'] = 2022
income_2023['year'] = 2023
income_2024['year'] = 2024

#consolidate all files
income = pd.concat([income_2020, income_2021, income_2022, income_2023, income_2024], ignore_index=True)
income.dtypes
income = income.rename(columns={'Label (Grouping)': 'Income Range'})
income = income.loc[:, ~income.columns.str.contains("Margin of Error", case=False, na=False)]

In [56]:
# Data cleaning
def percent_to_decimal(x):
    if isinstance(x, str) and "%" in x:
        # remove % and commas, convert to decimal
        return pd.to_numeric(x.replace("%", "").replace(",", ""), errors="coerce") / 100
    elif isinstance(x, str):
        # remove commas and convert to numeric
        return pd.to_numeric(x.replace(",", ""), errors="coerce")
    else:
        return x

cols_to_convert = [col for col in income.columns if col not in ["year", "Income Range"]]
income[cols_to_convert] = income[cols_to_convert].apply(lambda col: col.map(percent_to_decimal))

#filter only household columns
household_cols = [col for col in income.columns if "Households!!Estimate" in col]
# Add year + label columns
selected_cols = ["Income Range", "year"] + household_cols

# Create filtered dataframe
df_households = income[selected_cols]

def clean_household_name(col):
    if "Households!!Estimate" not in col:
        return col

    name = col.replace("!!Households!!Estimate", "")
    name = name.replace(" County, Minnesota", "")
    name = name.replace(", Minnesota", "")
    name = name.strip().rstrip(",")

    if name == "Minnesota":
        name = "Minnesota Total"
    return name.strip().lstrip("")

df_households = df_households.rename(columns={
    col: clean_household_name(col)
    for col in df_households.columns
})
# Show result
df_households.head(5)

,Income Range,year,Minnesota Total,Aitkin,Anoka,Becker,Beltrami,Benton,Big Stone,Blue Earth,...,Traverse,Wabasha,Wadena,Waseca,Washington,Watonwan,Wilkin,Winona,Wright,Yellow Medicine
0,Total,2020,2207988.000,7594.000,129308.000,13942.000,17882.000,16482.000,2294.000,26390.000,...,1590.000,9081.000,5748.000,7547.000,95796.000,4371.000,2799.000,19466.000,49097.000,4088.000
1,"Less than $10,000",2020,0.041,0.062,0.023,0.053,0.069,0.048,0.049,0.046,...,0.040,0.032,0.056,0.047,0.024,0.049,0.038,0.040,0.027,0.047
2,"$10,000 to $14,999",2020,0.034,0.060,0.020,0.046,0.067,0.031,0.066,0.040,...,0.089,0.033,0.063,0.038,0.014,0.071,0.065,0.053,0.023,0.045
3,"$15,000 to $24,999",2020,0.070,0.108,0.052,0.078,0.104,0.063,0.123,0.097,...,0.076,0.095,0.110,0.105,0.042,0.117,0.076,0.099,0.047,0.088
4,"$25,000 to $34,999",2020,0.075,0.122,0.062,0.094,0.108,0.101,0.088,0.098,...,0.111,0.087,0.110,0.095,0.046,0.119,0.074,0.086,0.056,0.077


In [58]:
# Columns to keep as identifier
id_cols = ["year", "Income Range"]

# Columns to melt into 'variable' and 'value'
value_cols = [col for col in df_households.columns if col not in id_cols]

# Melt the dataframe
df_melted = df_households.melt(id_vars=id_cols, value_vars=value_cols, 
                        var_name="county", value_name="Value")
# Pivot table: 'year' stays as row, 'Income Range' becomes columns, values aggregated by mean
df_pivot = df_melted.pivot_table(
    index=['year', 'county'],
    columns='Income Range',
    values='Value',
    aggfunc='mean'
)
    
# Flatten multi-level columns
df_pivot.columns = [f"{col}" for col in df_pivot.columns]
df_pivot = df_pivot.reset_index()
df_pivot.columns = (
    df_pivot.columns
        .str.replace("\u00A0", " ", regex=False)  # non-breaking space
        .str.replace("\t", " ", regex=False)       # tabs
        .str.strip()
)
df_pivot = df_pivot.apply(lambda col: col.str.lower() if col.dtypes == 'object' else col)
df_pivot.to_csv("mn_income_clean.csv", index=False)
df_pivot.head(5)

,year,county,Mean income (dollars),Median income (dollars),Total,"$10,000 to $14,999","$100,000 to $149,999","$15,000 to $24,999","$150,000 to $199,999","$200,000 or more","$25,000 to $34,999","$35,000 to $49,999","$50,000 to $74,999","$75,000 to $99,999",Household income in the past 12 months,"Less than $10,000"
0,2020,aitkin,63678.0,49086.0,7594.0,0.060,0.113,0.108,0.031,0.027,0.122,0.155,0.205,0.119,0.318,0.062
1,2020,anoka,100418.0,84379.0,129308.0,0.020,0.218,0.052,0.102,0.076,0.062,0.109,0.169,0.169,0.289,0.023
2,2020,becker,84677.0,60508.0,13942.0,0.046,0.147,0.078,0.044,0.060,0.094,0.140,0.195,0.145,0.310,0.053
3,2020,beltrami,66653.0,50525.0,17882.0,0.067,0.119,0.104,0.044,0.035,0.108,0.148,0.184,0.123,0.285,0.069
4,2020,benton,75804.0,60564.0,16482.0,0.031,0.150,0.063,0.060,0.032,0.101,0.172,0.177,0.166,0.374,0.048


## _2. MN Crime Data_
Source:

County and Municipal Offense Information by County Data | Minnesota Crime Data Explorer | Bureau of Criminal Apprehension

•	OffenseCountyMunicipalByCounty 2021-2025

State Population by Characteristics: 2020-2024
MN demography: https://www2.census.gov/programs-surveys/popest/datasets/2020-2024/counties/asrh/cc-est2024-agesex-13.csv


In [61]:
#  Minnesota Crime Data County and Municipal Offense Information by County Data
mn_crime_2021 = pd.read_excel("OffenseCountyMunicipalByCounty 2021.xlsx")
mn_crime_2022 = pd.read_excel("OffenseCountyMunicipalByCounty 2022.xlsx")
mn_crime_2023 = pd.read_excel("OffenseCountyMunicipalByCounty 2023.xlsx")
mn_crime_2024 = pd.read_excel("OffenseCountyMunicipalByCounty 2024.xlsx")
mn_crime_2025 = pd.read_excel("OffenseCountyMunicipalByCounty 2025.xlsx")
mn_crime_2021['year']= '2021'
mn_crime_2022['year']= '2022'
mn_crime_2023['year']= '2023'
mn_crime_2024['year']= '2024'
mn_crime_2025['year']= '2025'
# Combine (union) them
# Union and filter in one step
mn_crime = pd.concat([mn_crime_2021, mn_crime_2022, mn_crime_2023, mn_crime_2024, mn_crime_2025], ignore_index=True)
mn_crime = mn_crime[mn_crime['Statistic'] == 'Actual']

mn_crime_trend = pd.read_excel("MNHistoricalCrimeIndex 2025.xlsx")
mn_crime_trend.head(5)

,Index Year,Year Population,Crime Index Total,Murder,Rape,Robbery,Aggravated Assault,Burglary,Larceny,Motor Vehicle Theft,Arson,Crime Rate,Unnamed: 12
0,1936,2563953,16753,38,101,788,274,4778,7203,3571,0,654.6,653.405113
1,1937,2723798,17065,35,73,661,180,4000,8843,3273,0,626.5,626.514888
2,1938,2746633,19312,33,127,648,175,4203,10984,3142,0,703.1,703.115414
3,1939,2769468,20139,54,156,649,207,4665,11582,2826,0,727.2,727.179372
4,1940,2792300,19514,35,208,416,210,4967,11473,2205,0,698.9,698.850410


In [63]:
#Standardize string columns to lowercase.
mn_crime.columns = mn_crime.columns.str.lower()
mn_crime_trend.columns = mn_crime_trend.columns.str.lower()
mn_crime = mn_crime.apply(lambda col: col.str.lower() if col.dtypes == 'object' else col)
mn_crime_trend = mn_crime_trend.apply(lambda col: col.str.lower() if col.dtypes == 'object' else col)
mn_crime_trend["index year"] = pd.to_numeric(mn_crime_trend["index year"], errors="coerce").astype("Int64")

mn_crime.to_csv("mn_crime_clean.csv", index=False)
mn_crime.head(1)

,county,county population,statistic,aggravated assault,all other larceny,animal cruelty,arson,assisting or promoting prostitution,betting/wagering,bribery,...,stolen property offenses,theft from building,theft from coin-operated machine or device,theft from motor vehicle,theft of motor vehicle parts or accessories,weapon law violations,welfare fraud,wire fraud,group a total,year
0,aitkin,15849.0,actual,7.0,106.0,1.0,0.0,0.0,0.0,0.0,...,2.0,2.0,0.0,17.0,11.0,18.0,0.0,1.0,694.0,2021
